**CI twin of `ch04-generalization-splits-cv.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

homes = load_csv("california-housing-sample")
features = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]
X, y = homes[features], homes["MedHouseVal"]

train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"training houses: {len(train_X)}   held-out houses: {len(test_X)}")

# The parrot: a lookup table of every training house, mean price otherwise.
price_book = {tuple(row): price
              for row, price in zip(train_X.itertuples(index=False), train_y)}
fallback = train_y.mean()

def parrot(rows):
    return [price_book.get(tuple(r), fallback)
            for r in rows.itertuples(index=False)]

print(f"parrot MAE on training houses: "
      f"{mean_absolute_error(train_y, parrot(train_X)):.3f}")
print(f"parrot MAE on unseen houses:   "
      f"{mean_absolute_error(test_y, parrot(test_X)):.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression

for cols, label in [(["MedInc"], "1 feature "), (features, "8 features")]:
    model = LinearRegression().fit(train_X[cols], train_y)
    train_mae = mean_absolute_error(train_y, model.predict(train_X[cols]))
    test_mae = mean_absolute_error(test_y, model.predict(test_X[cols]))
    print(f"{label}:  train MAE {train_mae:.3f}   test MAE {test_mae:.3f}")

In [ ]:
from sklearn.metrics import root_mean_squared_error

model8 = LinearRegression().fit(train_X, train_y)
pred = model8.predict(test_X)

print(f"test MAE:  {mean_absolute_error(test_y, pred):.3f}")
print(f"test RMSE: {root_mean_squared_error(test_y, pred):.3f}")

In [ ]:
for seed in (0, 1, 2, 3, 4):
    a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = LinearRegression().fit(a, c)
    print(f"seed {seed}: test MAE {mean_absolute_error(d, m.predict(b)):.3f}")

In [ ]:
from sklearn.model_selection import cross_val_score

scores = -cross_val_score(LinearRegression(), X, y, cv=5,
                          scoring="neg_mean_absolute_error")
print("fold MAEs:", [round(s, 3) for s in scores])
print(f"report: {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
leaky = homes.copy()
# Feels like innocent feature engineering — but MedHouseVal IS the target.
leaky["PricePerRoom"] = leaky["MedHouseVal"] / leaky["AveRooms"]

Xl = leaky[features + ["PricePerRoom"]]
a, b, c, d = train_test_split(Xl, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(a, c)
print(f"test MAE with the leaky feature: "
      f"{mean_absolute_error(d, m.predict(b)):.3f}")

In [ ]:
homes = load_csv("california-housing-sample")
X, y = homes[["MedInc"]], homes["MedHouseVal"]

train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(train_X, train_y)
test_mae = mean_absolute_error(test_y, model.predict(test_X))

run_tests([
    ("fit saw only training rows", round(
        mean_absolute_error(train_y, model.predict(train_X)), 3), 0.64),
    ("honest test MAE", round(test_mae, 3), 0.698),
])

In [ ]:
def k_fold_indices(n, k):
    folds = []
    start = 0
    for fold in range(k):
        size = n // k + (1 if fold < n % k else 0)
        test_ids = list(range(start, start + size))
        train_ids = [i for i in range(n) if i < start or i >= start + size]
        folds.append((train_ids, test_ids))
        start += size
    return folds

run_tests([
    ("number of folds", len(k_fold_indices(7, 3)), 3),
    ("fold 1 test rows", k_fold_indices(7, 3)[0][1], [0, 1, 2]),
    ("fold 2 test rows", k_fold_indices(7, 3)[1][1], [3, 4]),
    ("fold 3 test rows", k_fold_indices(7, 3)[2][1], [5, 6]),
    ("fold 2 train rows", k_fold_indices(7, 3)[1][0], [0, 1, 2, 5, 6]),
    ("even split test rows", k_fold_indices(6, 3)[2][1], [4, 5]),
])